In [9]:
#| default_exp machine_learning.train_split

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-21. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Every ML pipeline in this workspace needs the same train/validation split of the vault's notes, and the split must be **reproducible**: regenerating a dataset must never leak training items into validation.

Python's built-in `hash()` cannot provide this — it is salted per-process (`PYTHONHASHSEED`), so membership silently reshuffles on every kernel restart. (A dataset-consolidation section of ~150 lines exists in one playground notebook solely to undo that damage.) These helpers use an MD5-based digest instead, and split by a stable identifier (typically the note *name*), so one note belongs to exactly one side for **every** task that uses this split.

In [10]:
#| export
import hashlib
from typing import Callable

In [11]:
from fastcore.test import *

In [12]:
#| export
def get_stable_hash_modulo(identifier: str, modulo: int = 100) -> int:
    """
    Return a deterministic integer in `[0, modulo)` for `identifier`.

    Uses MD5 rather than Python's built-in `hash()`, which is salted
    per-process (`PYTHONHASHSEED`) and therefore changes on every kernel
    restart.

    **Parameters**
    - `identifier` (*str*) — The string to hash (e.g. a note name).
    - `modulo` (*int*) — Size of the residue space.
    """
    clean_id = identifier.strip().lower()
    return int(hashlib.md5(clean_id.encode('utf-8')).hexdigest(), 16) % modulo

In [13]:
#| hide
names = [f'note_{i}' for i in range(200)]

# Determinism: repeated calls give identical buckets, all in range.
buckets = [get_stable_hash_modulo(n) for n in names]
test_eq(buckets, [get_stable_hash_modulo(n) for n in names])
assert all(0 <= b < 100 for b in buckets)

# Normalization: case and surrounding whitespace do not change the bucket.
test_eq(get_stable_hash_modulo('MyNote'), get_stable_hash_modulo('  mynote '))

# `modulo` bounds the range.
assert all(0 <= get_stable_hash_modulo(n, modulo=7) < 7 for n in names)

In [14]:
#| export
def split_list_consistently(
        items: list, # The items to split.
        key: Callable = str, # Maps an item to its stable identifier, e.g. `lambda note: note.name`.
        valid_percentage: int = 20 # Residue-space percentage (`[0, 100)`) reserved for validation.
        ) -> tuple[list, list]:
    """
    Split `items` into `(valid_set, train_set)` deterministically.

    An item is in the validation set iff
    `get_stable_hash_modulo(key(item)) < valid_percentage`, so membership
    is stable across runs and machines — and global, since tasks splitting
    on the same identifiers agree on which items are held out.

    **Parameters**
    - `items` (*list*) — The items to split.
    - `key` (*Callable*) — Maps an item to its stable identifier.
    - `valid_percentage` (*int*) — Percentage of the residue space reserved for validation.
    """
    valid_set = [i for i in items if get_stable_hash_modulo(key(i)) < valid_percentage]
    train_set = [i for i in items if get_stable_hash_modulo(key(i)) >= valid_percentage]
    return valid_set, train_set

In [15]:
#| hide
# Split is disjoint, covering, and stable across calls.
valid, train = split_list_consistently(names)
test_eq(sorted(valid + train), sorted(names))
test_eq(set(valid) & set(train), set())
test_eq(split_list_consistently(names), (valid, train))

# The default 20% target lands in a sane band for this sample.
assert 5 <= 100 * len(valid) / len(names) <= 40

# `valid_percentage=0` reserves nothing; membership only grows with the percentage.
test_eq(split_list_consistently(names, valid_percentage=0), ([], names))
valid2, _ = split_list_consistently(names, valid_percentage=40)
assert set(valid) <= set(valid2)

## Examples

In [16]:
note_names = ['_index_alper', '_index_baum_fulton', 'smooth_morphism', 'gerbe_coherence']
valid, train = split_list_consistently(note_names)
print('valid:', valid)
print('train:', train)

valid: ['_index_alper']
train: ['_index_baum_fulton', 'smooth_morphism', 'gerbe_coherence']
